# ДЗ 3. Дообучение BERT для распознавания именованных сущностей (NER)

---

### Зачем это нужно

Извлечение именованных сущностей (Named Entity Recognition, NER) —  людей, организаций, географических объектов и других сущностей —  это один из первых шагов превращения неструктурированного текста в структурированные данные. На этой задаче строится поиск по сущностям, аналитика, извлечение информации и базы знаний.

Мы разобрали, как предобученную модель BERT можно адаптировать к новой задаче, заменив выходную классификационную голову и дообучив модель на размеченных данных. В этом домашнем задании вы проделаете этот процесс самостоятельно: возьмете предобученный русскоязычный BERT и дообучите его для задачи разметки последовательностей.

Главный результат работы —  не достижение высоких значений метрики, а корректная и осмысленная оценка качества модели. Вы увидите, почему в NER оценки на уровне токенов могут вводить в заблуждение, научитесь интерпретировать метрики на уровне сущностей и анализировать типичные ошибки модели.

### Что вы сделаете

1. Загрузите русскоязычную часть датасета WikiANN и разберетесь с BIO-разметкой и дисбалансом класса `O` (блок 1).
2. Разберетесь, как BIO-метки слов сопоставляются с подсловными токенами BERT (блок 2).
3. Соберете модель AutoModelForTokenClassification и дообучите ее на задаче NER (блок 3).
4. Сравните метрики token-level и entity-level и объясните, почему они могут существенно различаться (блок 4).
5. Проанализируете ошибки модели: неверные границы сущностей, ошибки в определении типа и особенности, связанные с подсловной токенизацией (блок 5).
6. Сформулируете обоснованные выводы по результатам эксперимента (блок 6).

> **Сколько займет:** ориентировочно 4— 6 часов.

> **Про объем кода:** не превращайте ноутбук в полотно, пишите функциями, комментируйте код. Лаконичное и читаемое решение оценивается выше длинного и запутанного.

> **Требуется GPU.** В Google Colab выберите Runtime → Change runtime type → GPU. Без CPU обучение модели будет занимать слишком много времени.

## Настройка окружения

In [ ]:
!pip install -q -U transformers datasets seqeval accelerate

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)
from seqeval.metrics import (
    classification_report as seq_classification_report,
    f1_score as seq_f1,
    precision_score as seq_precision,
    recall_score as seq_recall,
)
from sklearn.metrics import classification_report as sk_classification_report

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)

MODEL = "cointegrated/rubert-tiny2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)
if device == "cpu":
    print("GPU не найден, рекомендуется включить GPU.")

## 1. Данные и BIO-разметка (1 балл)

**Датасет:** [`WikiANN (ru)`](https://huggingface.co/datasets/unimelb-nlp/wikiann) размечен по трем типам сущностей —  `PER` (люди), `ORG` (организации), `LOC` (локации) —  в схеме BIO. Каждый пример —  это список слов (`tokens`) и список меток (`ner_tags`) той же длины. Ячейка ниже грузит датасет.

In [ ]:
raw = load_dataset("unimelb-nlp/wikiann", "ru")

# Берем подвыборку (полный train не нужен для целей ДЗ)
raw["train"] = raw["train"].shuffle(seed=RANDOM_STATE).select(range(8000))
raw["validation"] = raw["validation"].shuffle(seed=RANDOM_STATE).select(range(2000))
raw["test"] = raw["test"].shuffle(seed=RANDOM_STATE).select(range(2000))

# Имена меток: индекс = id метки
label_names = raw["train"].features["ner_tags"].feature.names
id2label = {i: t for i, t in enumerate(label_names)}
label2id = {t: i for i, t in enumerate(label_names)}

print("Метки:", label_names)
print("Размеры:", {k: len(v) for k, v in raw.items()})

# Как выглядит один пример:
example = raw["train"][1]
print("\nПример:")
for w, t in zip(example["tokens"], example["ner_tags"]):
  print(f"{w:<20} {id2label[t]}")

### ✍️ Задание 1.1. Распределение меток и длин предложений

По обучающей выборке `raw["train"]` выполните следующие действия:

1. Постройте столбчатую диаграмму распределения BIO-меток (количество токенов каждой метки).
2. Постройте гистограмму распределения длин предложений (в токенах).
3. Вычислите долю токенов с меткой `O` относительно общего числа токенов.

Требования: у каждого графика должны быть подписаны оси и указан заголовок.

>*Подсказка.* Все метки можно собрать в один список:
>```
>all_tags = [t for row in raw["train"]["ner_tags"] for t in row]
>```
>После этого преобразуйте идентификаторы меток в их названия с помощью словаря `id2label`.

In [ ]:
# ✍️ ВАШ КОД (задание 1.1)
# 1) bar plot распределения BIO-меток
# 2) гистограмма длины предложений в токенах
# 3) доля токенов с меткой "O"

### 💬 Задание 1.2. Выводы (впишите ответ ниже)

Ответьте своими словами, обязательно опираясь на числа, полученные в вашем эксперименте.

1. Какова доля токенов с меткой `O`? Какой тип сущностей (PER, ORG или LOC) встречается реже всего?
2. Что можно сказать о распределении длин предложений? Есть ли большинство коротких или длинных предложений, встречаются ли выбросы?
3. Почему при такой доле токенов `O` token-level accuracy плохо отражает качество модели NER? На какую метрику следует ориентироваться вместо нее и почему?

> *Ваш ответ:*
>
> 1. ...
> 2. ...
> 3. ...

## 2. Токенизация и выравнивание меток (1 балл)

В датасете BIO-метки заданы на уровне слов, тогда как BERT работает с подсловными токенами (subwords). Поэтому перед обучением необходимо перенести метки слов на последовательность подсловных токенов.

В ячейке ниже приведена готовая функция, которая выполняет это выравнивание. Разберитесь, как она работает: в задании 2.1 вам потребуется объяснить ее логику своими словами.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

def align_labels_with_tokens(labels, word_ids):
  """Переносит пословные BIO-метки на subwords."""
  new_labels = []
  prev_word = None
  for word_id in word_ids:
      if word_id is None: # спецтокены [CLS] / [SEP] / паддинг
          new_labels.append(-100)
      elif word_id != prev_word: # первый subword нового слова
          new_labels.append(labels[word_id])
      else: # продолжение того же слова
          new_labels.append(-100)
      prev_word = word_id
  return new_labels

def tokenize_and_align(batch):
  tok = tokenizer(batch["tokens"], truncation=True, is_split_into_words=True, max_length=128)
  tok["labels"] = [align_labels_with_tokens(labels, tok.word_ids(batch_index=i)) for i, labels in enumerate(batch["ner_tags"])]
  return tok

tokenized = raw.map(tokenize_and_align, batched=True, remove_columns=raw["train"].column_names)
data_collator = DataCollatorForTokenClassification(tokenizer)

# Демонстрация на примере
example = raw["train"][1]
enc = tokenizer(example["tokens"], is_split_into_words=True)
aligned = align_labels_with_tokens(example["ner_tags"], enc.word_ids())
print(f'{"токен":<16}{"метка":<10}')
for tok_str, lab in zip(tokenizer.convert_ids_to_tokens(enc["input_ids"]), aligned):
  print(f'{tok_str:<16}{("-100" if lab == -100 else id2label[lab]):<10}')

### 💬 Задание 2.1. Вопрос на понимание (впишите ответ ниже)

Изучите вывод примера выше и ответьте своими словами (3— 5 предложений):

1. Почему метка сущности присваивается только первому subword слова, а остальные получают значение `-100`?
2. Какую роль играет значение `-100` при обучении модели? Почему его также используют для специальных токенов `[CLS]` и `[SEP]`?
3. Какие проблемы возникли бы, если бы метку слова присваивали всем его subwords?

> *Ваш ответ:* ...

## 3. Дообучение модели (2 балла)

Для оценки качества модели в задаче NER используются метрики на уровне сущностей (entity-level), которые вычисляются с помощью библиотеки seqeval. Сущность считается распознанной правильно только в том случае, если модель верно определила и ее границы, и ее тип.

Функция ниже будет вычислять precision, recall и F1 после каждой эпохи обучения. Используйте ее без изменений.

In [ ]:
def compute_metrics(eval_pred):
  logits, labels = eval_pred
  preds = np.argmax(logits, axis=-1)

  true_tags, pred_tags = [], []
  for pred_row, label_row in zip(preds, labels):
      tt, pp = [], []
      for p_i, l_i in zip(pred_row, label_row):
          if l_i == -100:
              continue
          tt.append(id2label[l_i])
          pp.append(id2label[p_i])
      true_tags.append(tt)
      pred_tags.append(pp)

  return {
      "precision": seq_precision(true_tags, pred_tags),
      "recall":    seq_recall(true_tags, pred_tags),
      "f1":        seq_f1(true_tags, pred_tags),
  }

### ✍️ Задание 3.1. Дообучите модель

1. Загрузите модель с помощью `AutoModelForTokenClassification.from_pretrained`, указав корректные значения `num_labels`, `id2label` и `label2id`.
2. Настройте `TrainingArguments`: оценка на валидационной выборке после каждой эпохи; 3–4 эпохи обучения; `batch_size=16`; `learning_rate=5e-5`; `seed=RANDOM_STATE`.
3. Создайте `Trainer`, передав обучающую и валидационную выборки, `data_collator`, `compute_metrics` и остальные необходимые параметры, затем запустите обучение с помощью `trainer.train()`.

Во время обучения значение eval_f1 должно постепенно расти.

Сохраните объект `Trainer` в переменную `trainer` —  он понадобится в следующих заданиях.

In [ ]:
# ✍️ ВАШ КОД (задание 3.1)

## 4. Метрики token-level и entity-level (2 балла)

Функция ниже запускает обученную модель на указанном датасете и возвращает две последовательности строковых меток: истинные и предсказанные. Они уже приведены к формату, который можно напрямую использовать для вычисления как entity-level-метрик с помощью seqeval, так и token-level-метрик с помощью sklearn.

Это позволит сравнить оба подхода к оценке качества модели и понять, почему в задаче NER они могут давать заметно разные результаты.

In [ ]:
def get_tags(trainer, dataset):
  out = trainer.predict(dataset)
  preds = np.argmax(out.predictions, axis=-1)
  labels = out.label_ids

  true_tags, pred_tags = [], []
  for pred_row, label_row in zip(preds, labels):
      tt, pp = [], []
      for p_i, l_i in zip(pred_row, label_row):
          if l_i == -100:
              continue
          tt.append(id2label[l_i])
          pp.append(id2label[p_i])
      true_tags.append(tt)
      pred_tags.append(pp)
  return true_tags, pred_tags

### ✍️ Задание 4.1. Сравните две оценки на тестовой выборке

1. Получите последовательности истинных и предсказанных меток для тестовой выборки.
2. Выведите entity-level-отчет с помощью `seq_classification_report(true_tags, pred_tags, digits=3)`.
3. Выведите token-level-отчет. Для этого разделите обе последовательности в плоские списки и передайте их в `sk_classification_report(...)`.

In [ ]:
# ✍️ ВАШ КОД (задание 4.1)

### 💬 Задание 4.2 — объясните разрыв (впишите ответ ниже)

Задание 4.2. Объясните различия между двумя оценками

Ответьте своими словами, обязательно используя значения из двух полученных отчетов.

1. Чему равны token-level accuracy и entity-level F1? Насколько они отличаются?
2. Назовите две причины, по которым token-level accuracy оказывается заметно выше.
3. Какую из этих двух оценок вы бы использовали как основную характеристику качества NER-модели и почему?

> *Ваш ответ:*
>
> 1. ...
> 2. ...
> 3. ...

## 5. Анализ ошибок по сущностям (2 балла)

Функция `get_tags` возвращает метки на уровне слов в том же порядке, в котором слова представлены в `raw["test"]["tokens"]`. Благодаря этому каждому слову соответствует ровно одна истинная и одна предсказанная метка, поэтому слова и теги можно сопоставлять напрямую при анализе ошибок модели.

### ✍️ Задание 5.1. Найдите и выведите ошибки модели

Просмотрите предложения тестовой выборки и найдите те, в которых предсказанные метки (`pred_tags`) отличаются от истинных (`true_tags`).

Выведите не менее трех таких предложений. Для каждого покажите:

- слово, в котором обнаружено расхождение;
- истинную метку;
- предсказанную метку;
- несколько соседних слов для контекста (до и после ошибки).

In [ ]:
# ✍️ ВАШ КОД (задание 5.1)

### 💬 Задание 5.2. Проанализируйте ошибки модели (впишите ответ ниже)

Опираясь на найденные вами примеры, ответьте на вопросы:

1. Отнесите найденные ошибки как минимум к двум разным типам: нарушение границ сущности, путаница типа между PER, ORG и LOC, пропуск сущности или ложное срабатывание.
2. Для каждого типа приведите конкретный пример из своего вывода.
Выберите один из наиболее часто встречающихся типов ошибок и объясните, почему он возникает. Объяснение должно быть связано с особенностями текста или данных (например, неоднозначностью контекста, похожими шаблонами, редкостью примеров и т.п.).

> *Ваш ответ:*
>
> 1. ...
> 2. ...

## 6. Вывод (2 балла)

### 💬 Задание 6.1. Вывод (впишите ответ ниже)

Напишите вывод объемом 5—8 предложений, опираясь на результаты своего эксперимента.

В выводе ответьте на следующие вопросы:

1. Какое значение entity-level F1 удалось получить? Какие типы сущностей (PER, ORG, LOC) распознаются лучше, а какие хуже? Подкрепите ответ результатами и предложите возможное объяснение.
2. Для каких задач такая модель уже может быть полезна? В каких сценариях ее качества пока недостаточно?
3. Какие ограничения были у проведенного эксперимента (например, шум разметки, размер обучающей выборки, размер модели, ограничение длины последовательности)? Назовите одно конкретное улучшение, которое, по вашему мнению, может повысить качество модели.

> *Ваш ответ:* ...

## Бонусное задание

> Бонусная часть не является обязательной к выполнению и не влияет на оценку за домашнее задание. Выполнив это задание, вы получите развернутую обратную связь в свободной форме. Здесь нет строгих критериев выполнения.

В основном задании вы дообучали всю модель: обновлялись веса и энкодера BERT, и классификационной головы. Однако существует более дешевый режим обучения —  заморозить энкодер и обучать только выходную голову поверх фиксированных контекстных представлений, которые вычисляет BERT.

Такой подход позволяет значительно сократить число обучаемых параметров и ускорить обучение, но обычно приводит к некоторой потере качества.

Проверьте это на практике: сравните полное дообучение модели с обучением только классификационной головы и оцените, насколько меняются качество и время обучения.

### ✍️ Задание Б1. Обучите модель с замороженным энкодером

1. Загрузите новую модель с помощью `AutoModelForTokenClassification.from_pretrained`, используя те же параметры, что и в блоке 3.
2. Заморозьте энкодер BERT: пройдите по `model_frozen.base_model.parameters()` и установите `requires_grad=False` для всех его параметров. Классификационная голова (`classifier`) должна остаться обучаемой.
3. Создайте новый объект `Trainer` (можно использовать те же `TrainingArguments`), обучите модель и сравните ее с моделью из основного задания: значение entity-level F1 на тестовой выборке и время обучения.

> *Подсказка.* Время обучения удобно измерить с помощью `time.time()` до и после вызова `trainer.train()`.

In [ ]:
# ✍️ ВАШ КОД (задание Б1)

### 💬 Задание Б2. Рефлексия (впишите ответ ниже)

Ответьте своими словами, опираясь на результаты своего эксперимента.

1. Как изменились entity-level F1 и время обучения после заморозки энкодера? Приведите оба значения.
2. Почему заморозка энкодера обычно сильнее влияет на качество в задаче NER, чем в задаче классификации всего текста?
3. В каких случаях вы бы все же предпочли обучать только классификационную голову?

> *Ваш ответ:*
>
> 1. ...
> 2. ...
> 3. ...